**What is a UDF?**

`A UDF (User Defined Function) lets you write custom Python logic
and apply it column-wise in a Spark DataFrame. Spark doesn't have
built-in functions for every business rule — UDFs fill that gap.`

`▸ Runs per-row (like a map operation)
▸ Must be registered before use
▸ Can be used in DataFrame API or Spark SQL
▸ Has serialization overhead (data moves Python ↔ JVM)`

Basic UDF (DataFrame API)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# ── Sample data ──
data = [
    (1, "aarav",  "mehta",  45000),
    (2, "priya",  "sharma", 62000),
    (3, "rohan",  "patel",  38000),
]
df = spark.createDataFrame(data, ["id", "first", "last", "salary"])

def format_name(first, last):
  """Capitalize and combine first and last name"""
  return f"{first.strip().title()} {last.strip().title()}"

  format_name_udf = udf(format_name, StringType())

  df.withColumn("full_name", format_name_udf(col("first"), col("last"))) \
    .show()

In [0]:
from pyspark.sql.types import DoubleType

# ── Indian tax slab logic (simplified FY 2024-25) ──
def calculate_tax(salary):
    if salary is None:
        return 0.0
    annual = salary * 12
    if annual <= 300000:
        return 0.0
    elif annual <= 600000:
        return (annual - 300000) * 0.05
    elif annual <= 900000:
        return 15000 + (annual - 600000) * 0.10
    elif annual <= 1200000:
        return 45000 + (annual - 900000) * 0.15
    else:
        return 90000 + (annual - 1200000) * 0.20

tax_udf = udf(calculate_tax, DoubleType())

df.withColumn("annual_tax", tax_udf(col("salary"))) \
  .show()

In [0]:
# ── Register so you can use in SQL queries ──
spark.udf.register("calc_tax", calculate_tax, DoubleType())

df.createOrReplaceTempView("employees")

spark.sql("""
    SELECT id, first, last, salary,
           calc_tax(salary) AS annual_tax
    FROM employees
    WHERE calc_tax(salary) > 10000
""").show()

Decorator Syntax

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# ── Cleaner syntax with decorator ──
@udf(returnType=StringType())
def mask_email(email):
    """Mask email for GDPR compliance"""
    if email is None or "@" not in email:
        return "***"
    user, domain = email.split("@")
    masked = user[0] + "***" + user[-1]
    return f"{masked}@{domain}"

emails = spark.createDataFrame([
    ("aarav@gmail.com",),
    ("priya.s@company.co.in",),
], ["email"])

emails.withColumn("masked", mask_email(col("email"))).show()

**3 Ways to Call UDFs**
`Once a UDF is defined, there are three primary ways to invoke it:`

 `DataFrame API   — df.withColumn("col", my_udf(col("x")))`
 `Spark SQL       — spark.udf.register() → SQL query`
 `selectExpr      — df.selectExpr("my_udf(col) AS result")`

`Each method has trade-offs in readability, performance,`
`and where it can be used (notebooks vs jobs vs SQL warehouses).`